# 動画生成アプリ用 Colabノートブック（ComfyUI + MiniMax H3）

このノートブックは、Androidアプリの接続先になります。

1. ランタイムのタイプを **GPU（L4 または A100）** にする（T4は不可）
2. 上から順に実行する（「すべてを実行」でOK）
3. 最後のセルは動かし続ける。止めるとアプリから見て「停止中」になる

起動すると、ドライブの `ColabVideoApp/status.json` にURLや残量が書き出されます。アプリはこのファイルを読みます。

In [ ]:
#@title 0. 設定（必要なら変更）
DRIVE_DIR = "/content/drive/MyDrive/ColabVideoApp"   # アプリが読むフォルダ
USE_SAGE = False            # L4で高解像度(960×1728以上)を作るなら False のまま
LLM_ID = "Qwen/Qwen2.5-7B-Instruct"   # ランダム生成・英訳に使うLLM
HF_TOKEN_SECRET_NAME = "HF_TOKEN"     # 必要なときだけ、Colabのシークレットに登録

# モデルをGoogleドライブに保存しておき、次回からそこから読み込む（ダウンロードを省く）
#   "off" = 保存しない（毎回ダウンロード。標準）
#   "llm" = LLMだけ保存（約15GB） / "all" = 全モデルとLLMを保存（約45GB）
DRIVE_CACHE = "off"

# 使用量の表示用（Colabの「リソースを表示」で確認した値を入れる）
UNITS_PER_HOUR = None   # 例: 1.9 （今のGPUの1時間あたり消費ユニット）
START_UNITS = None      # 例: 62.4 （起動時点の残りユニット。入れると概算の残量を出す）

In [ ]:
#@title 1. 環境チェックとドライブ接続
import os, sys, re, json, time, shutil, secrets, random, subprocess, threading, datetime
import torch

def sh(cmd, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout.strip():
        print(r.stdout.strip()[-1500:])
    if r.returncode != 0:
        print(r.stderr.strip()[-1500:])
        if check:
            raise RuntimeError("コマンドが失敗しました: " + cmd)
    return r

def gpu_key(name):
    for k in ("H100", "A100", "L4", "T4"):
        if k in name:
            return k
    return name

def snap_frames(seconds, fps=24):
    # H3のフレーム数は 17k+5 の並びに丸められる
    k = max(0, round((seconds * fps - 5) / 17))
    return 17 * k + 5

# 画質プリセット（実測: 2段生成 = 1段目で低解像度 → 2段目で2倍にハイレゾ化）
# stage1_mp: 1段目のメガピクセル / max_sec: そのGPUで安定して作れる最大秒数
PRESETS = {
    # est_min は Veda オン・4step の目安。実測: L4 960×1728 10秒=12分 / L4 フルHD8秒=11分 / A100 最高10秒=9分
    # A100 の高画質・フルHDは未実測の見込み値（L4 標準704×1280 5秒 = 約2.6〜3.8分を実測）
    "standard": dict(label="標準 704×1280", stage1_mp=0.22, out_short=704, out_long=1280,
                     gpus=["L4", "A100", "H100"], max_sec={"L4": 10, "A100": 15, "H100": 15},
                     est_min={"L4": 3, "A100": 2, "H100": 2}, measured=True),
    "high": dict(label="高画質 960×1728", stage1_mp=0.4, out_short=960, out_long=1728,
                 gpus=["L4", "A100", "H100"], max_sec={"L4": 10, "A100": 15, "H100": 15},
                 est_min={"L4": 12, "A100": 5, "H100": 4}, measured=True),
    "fullhd": dict(label="フルHD 1088×1920", stage1_mp=0.5, out_short=1088, out_long=1920,
                   gpus=["L4", "A100", "H100"], max_sec={"L4": 8, "A100": 15, "H100": 15},
                   est_min={"L4": 11, "A100": 6, "H100": 5}, measured=True),
    "max": dict(label="最高 1536×2752", stage1_mp=1.0, out_short=1536, out_long=2752,
                gpus=["A100", "H100"], max_sec={"A100": 10, "H100": 10},
                est_min={"A100": 9, "H100": 6}, measured=True),
}

if not torch.cuda.is_available():
    raise RuntimeError("GPUが有効ではありません。ランタイム > ランタイムのタイプを変更 から GPU を選んでください。")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_KEY = gpu_key(GPU_NAME)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
CAP = torch.cuda.get_device_capability(0)
if CAP[0] < 8:
    raise RuntimeError("このGPUはbf16に対応していません（T4など）。L4かA100に変えてください。")
if VRAM_GB < 20:
    raise RuntimeError("GPUメモリが足りません（20GB以上が必要）。")
print(f"GPU: {GPU_NAME} / {VRAM_GB:.1f}GB")
if GPU_KEY == "L4" and USE_SAGE:
    print("注意: L4でSageAttentionを使うと、960×1728以上の2段目でComfyUIが落ちます。USE_SAGE=False を推奨。")

free_gb = shutil.disk_usage("/content").free / 1024 ** 3
print(f"空きディスク: {free_gb:.0f}GB")
if free_gb < 35:
    raise RuntimeError("空きディスクが35GB未満です。")

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
STATUS_PATH = os.path.join(DRIVE_DIR, "status.json")
DRIVE_MODEL_DIRS = {"lora": os.path.join(DRIVE_DIR, "loras"), "checkpoint": os.path.join(DRIVE_DIR, "checkpoints")}
for _d in DRIVE_MODEL_DIRS.values():
    os.makedirs(_d, exist_ok=True)

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get(HF_TOKEN_SECRET_NAME)
except Exception:
    pass

In [ ]:
#@title 2. ComfyUIとカスタムノードの準備
C = "/content/ComfyUI"
if not os.path.exists(C):
    sh(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {C}")

# 実績のある構成（高速✕ハイレゾ化ノートブックと同じノード）
NODES = [
    "https://github.com/ltdrdata/ComfyUI-Manager",
    "https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git",        # Turbo
    "https://github.com/rgthree/rgthree-comfy",
    "https://github.com/kijai/ComfyUI-KJNodes.git",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "https://github.com/nicolab28/ComfyUI-ClipProj",                  # 軽量テキストエンコーダー
    "https://github.com/Tr1dae/ComfyUI-MiniMaxH3_LatentUpscaler.git", # 2段目ハイレゾ・高速VAE
    "https://github.com/mamad8c/ComfyUI-H3-Latent-Upscaler-Mamad8.git",
    "https://github.com/xmarre/ComfyUI-Spectrum-MiniMax-H3.git",      # Spectrum高速化
    "https://github.com/Luisacaotica/ComfyUI-MiniMaxH3Mod",           # RefMod（キャラ参照を小さなファイルにして再利用）
]
for url in NODES:
    name = url.split("/")[-1].replace(".git", "")
    path = f"{C}/custom_nodes/{name}"
    if os.path.exists(path) and not os.path.exists(path + "/.git"):
        shutil.rmtree(path)
    if not os.path.exists(path):
        sh(f"git clone --depth 1 {url} {path}", check=False)

# Veda スパースアテンション（L4実測で約3倍速・画質ほぼ同等。動作確認した版に固定）
def pinned(url, rev):
    d = f"{C}/custom_nodes/" + url.split("/")[-1][:-4]
    if not os.path.exists(f"{d}/__init__.py"):
        sh(f'rm -rf "{d}"; mkdir -p "{d}" && cd "{d}" && git init -q && git remote add origin {url} '
           f'&& git fetch -q --depth 1 origin {rev} && git checkout -q FETCH_HEAD', check=False)
    print("node:", os.path.basename(d), "OK" if os.path.exists(f"{d}/__init__.py") else "NG")
pinned("https://github.com/veda-sparse/Veda-on-ComfyUI.git", "8d7ac153ac215d636eaf1b5910b0536c3de9e6ed")

sh("apt-get install -y -qq aria2", check=False)
sh("pip install -q uv")
sh(f"uv pip install --system -r {C}/requirements.txt accelerate qwen-vl-utils 'transformers>=4.48.0' "
   "bitsandbytes comfy-aimdo comfy-kitchen sageattention gguf huggingface_hub websocket-client "
   "fastapi uvicorn requests hf_transfer safetensors --quiet")
sh("pip install -q -U comfy-kitchen gguf")
print("ComfyUIの準備ができました")

In [ ]:
#@title 3. モデルを準備（Driveに保存済みならそこから読み込み。なければダウンロード）
CACHE_DIR = os.path.join(DRIVE_DIR, "cache")
_put_threads = []

def drive_free_gb():
    try:
        return shutil.disk_usage("/content/drive").free / 1024 ** 3
    except Exception:
        return None

def cache_put(src, rel):
    # ダウンロード済みのファイルを、裏でドライブにも保存する（途中で止まっても壊れないよう .part で書いて最後に改名）
    if DRIVE_CACHE not in ("all",):
        return
    dst = os.path.join(CACHE_DIR, rel)
    size = os.path.getsize(src)
    if os.path.exists(dst) and os.path.getsize(dst) == size:
        return
    free = drive_free_gb()
    if free is not None and free < size / 1024 ** 3 + 1.0:
        print(f"ドライブの空きが足りないので保存しません: {rel}（空き {free:.0f}GB）")
        return
    def run():
        try:
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            t0 = time.time()
            shutil.copyfile(src, dst + ".part")
            os.replace(dst + ".part", dst)
            print(f"ドライブに保存しました: {rel}（{time.time() - t0:.0f}秒）")
        except Exception as e:
            print("ドライブへの保存に失敗:", rel, e)
    th = threading.Thread(target=run, daemon=True)
    th.start()
    _put_threads.append(th)

def cache_get(rel, dest):
    src = os.path.join(CACHE_DIR, rel)
    if DRIVE_CACHE != "all" or not os.path.exists(src) or os.path.getsize(src) < 10 * 1024 * 1024:
        return False
    t0 = time.time()
    shutil.copyfile(src, dest + ".part")
    os.replace(dest + ".part", dest)
    mb = os.path.getsize(dest) / 1024 ** 2
    print(f"ドライブから読み込み: {os.path.basename(dest)}（{mb / max(time.time() - t0, 0.1):.0f}MB/秒）")
    return True

MODELS = [
    # H3本体: FL2VA + Ref2VA + Turbo LoRA 統合版（t2v / i2v / ref2v を1つで担当）
    ("diffusion_models", "minimax_h3_fused_refdelta_r1024_turbo8_mystic07_int8_convrot.safetensors",
     "https://huggingface.co/MATLOWAI/minimax-h3-fused-turbo-int8-convrot/resolve/main/diffusion_models/minimax_h3_fused_refdelta_r1024_turbo8_mystic07_int8_convrot.safetensors"),
    # 軽量テキストエンコーダー（32B 15GB → 4B 4.5GB）
    ("text_encoders", "qwen3vl_4b_fp8_scaled.safetensors",
     "https://huggingface.co/Comfy-Org/Qwen3-VL/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors"),
    ("clip_projections", "mmh3-4b-ClipProj-v3.1-mlp.safetensors",
     "https://huggingface.co/NicoLab28/ClipProj-MiniMax-H3/resolve/main/mmh3-4b-ClipProj-v3.1-mlp.safetensors"),
    ("vae", "minimax_h3_audio_vae_fp32.safetensors",
     "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors"),
    ("vae", "minimax_h3_video_vae_int8_convrot.safetensors",
     "https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/main/minimax_h3_video_vae_int8_convrot.safetensors"),
    ("vae_approx", "taeh3.safetensors",
     "https://huggingface.co/Kijai/MiniMax-H3-TAE/resolve/main/vae_approx/taeh3.safetensors"),
    # Veda用の予測ファイル（高速化）
    ("veda", "minimax_h3_r2va_veda_preview_fp8.safetensors",
     "https://huggingface.co/Veda-Sparse/Minimax-H3-R2VA-Veda-Preview/resolve/1598b407905b7c0af85aa0c13acdfa6f01ff2982/minimax_h3_r2va_veda_preview_fp8.safetensors"),
    # 2段目のハイレゾ化に使う潜在アップスケーラー（約56MB）
    ("h3_latent_upscalers", "h3_clean_latent_upscaler_film_epoch200.safetensors",
     "https://huggingface.co/Tridae/H3LatentUpscaler/resolve/main/h3_clean_latent_upscaler_film_epoch200.safetensors"),
]
for subdir, fname, url in MODELS:
    d = f"{C}/models/{subdir}"
    os.makedirs(d, exist_ok=True)
    p = f"{d}/{fname}"
    if os.path.exists(p) and os.path.getsize(p) > 10 * 1024 * 1024:
        print("取得済み:", fname)
        continue
    if os.path.exists(p):
        os.remove(p)
    if cache_get(f"{subdir}/{fname}", p):
        continue
    print("ダウンロード中:", fname)
    hdr = f'--header="Authorization: Bearer {HF_TOKEN}"' if HF_TOKEN else ""
    sh(f'aria2c {hdr} -x 16 -s 16 -k 1M "{url}" -d "{d}" -o "{fname}"')
    if not os.path.exists(p):
        raise RuntimeError("ダウンロードに失敗しました: " + fname)
    cache_put(p, f"{subdir}/{fname}")

MODEL_FILES = {"diffusion": MODELS[0][1], "text_encoder": MODELS[1][1], "video_vae": MODELS[4][1]}
LORA_FILES = []
print("モデルの準備ができました")

In [ ]:
#@title 4. ComfyUIを起動
import requests
COMFY = "http://127.0.0.1:8188"
os.makedirs("/content/outputs", exist_ok=True)

cmd = [sys.executable, "main.py", "--port", "8188", "--enable-cors-header",
       "--output-directory", "/content/outputs",
       "--disable-pinned-memory", "--disable-comfy-compiler",
       "--preview-method", "auto", "--preview-size", "512"]
if USE_SAGE:
    cmd.append("--use-sage-attention")
comfy_proc = subprocess.Popen(cmd, cwd=C, stdout=open("/content/comfyui.log", "w"), stderr=subprocess.STDOUT)

ready = False
for _ in range(300):
    if comfy_proc.poll() is not None:
        break
    try:
        if requests.get(COMFY + "/system_stats", timeout=2).ok:
            ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    print(open("/content/comfyui.log").read()[-3000:])
    raise RuntimeError("ComfyUIが起動しませんでした。上のログを確認してください。")

nodes = requests.get(COMFY + "/object_info", timeout=60).json()
need = ["MiniMaxH3ReferenceToVideo", "MiniMaxH3ImageToVideo", "MiniMaxH3LatentUpscaleCombined",
        "SpectrumApplyMiniMaxH3", "MiniMaxH3VAEDecodeFast", "ClipProjApply", "VedaSparseAttention",
        "MiniMaxH3RefModExtract", "MiniMaxH3RefModsLoader", "MiniMaxH3RefModApply"]
for n in need:
    print(("OK  " if n in nodes else "なし ") + n)
print("ComfyUIが起動しました")

In [ ]:
#@title 5. プロンプト用LLMとヘルパーAPI
from transformers import AutoTokenizer, AutoModelForCausalLM
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import uvicorn

LLM_CACHE = os.path.join(CACHE_DIR, "llm", LLM_ID.replace("/", "__"))
_t0 = time.time()
if DRIVE_CACHE in ("llm", "all") and os.path.exists(os.path.join(LLM_CACHE, ".complete")):
    LLM_SRC = LLM_CACHE
    print("LLMをドライブから読み込みます:", LLM_CACHE)
else:
    # 初回は高速転送(hf_transfer)でダウンロードし、ドライブにも保存する
    try:
        import hf_transfer  # noqa
        os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    except Exception:
        pass
    from huggingface_hub import snapshot_download
    print("LLMをダウンロード中:", LLM_ID)
    LLM_SRC = snapshot_download(LLM_ID, local_dir="/content/llm", token=HF_TOKEN,
                                allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.tiktoken"])
    if DRIVE_CACHE in ("llm", "all"):
        def _save_llm():
            try:
                size = sum(os.path.getsize(os.path.join(LLM_SRC, f)) for f in os.listdir(LLM_SRC)
                           if os.path.isfile(os.path.join(LLM_SRC, f))) / 1024 ** 3
                free = drive_free_gb()
                if free is not None and free < size + 1.0:
                    print(f"ドライブの空きが足りないのでLLMは保存しません（必要 {size:.0f}GB / 空き {free:.0f}GB）")
                    return
                tmp = LLM_CACHE + ".part"
                shutil.rmtree(tmp, ignore_errors=True)
                os.makedirs(os.path.dirname(LLM_CACHE), exist_ok=True)
                t0 = time.time()
                shutil.copytree(LLM_SRC, tmp, ignore=shutil.ignore_patterns(".cache"))
                shutil.rmtree(LLM_CACHE, ignore_errors=True)
                os.replace(tmp, LLM_CACHE)
                open(os.path.join(LLM_CACHE, ".complete"), "w").write("ok")
                print(f"LLMをドライブに保存しました（{time.time() - t0:.0f}秒）")
            except Exception as e:
                print("LLMのドライブ保存に失敗:", e)
        threading.Thread(target=_save_llm, daemon=True).start()
tok = AutoTokenizer.from_pretrained(LLM_SRC)
llm = AutoModelForCausalLM.from_pretrained(LLM_SRC, torch_dtype=torch.bfloat16)  # 普段はCPUに置く
print(f"LLMの準備完了（{time.time() - _t0:.0f}秒）")
llm_lock = threading.Lock()

def comfy_busy():
    try:
        q = requests.get(COMFY + "/queue", timeout=5).json()
        return bool(q.get("queue_running")) or bool(q.get("queue_pending"))
    except Exception:
        return False

def chat(system, user, max_new_tokens=220, temperature=0.8):
    with llm_lock:
        # 動画モデルをGPUから降ろしてから、LLMをGPUに載せる
        try:
            requests.post(COMFY + "/free", json={"unload_models": True, "free_memory": True}, timeout=30)
        except Exception:
            pass
        llm.to("cuda")
        try:
            text = tok.apply_chat_template(
                [{"role": "system", "content": system}, {"role": "user", "content": user}],
                tokenize=False, add_generation_prompt=True)
            inputs = tok(text, return_tensors="pt").to("cuda")
            out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                               temperature=temperature, top_p=0.95)
            return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        finally:
            llm.to("cpu")
            torch.cuda.empty_cache()

PLACES = ["夕暮れの海辺の防波堤", "雨上がりの商店街", "霧のかかった森の小道", "夜の屋上", "雪の降る駅のホーム",
          "満開の桜並木", "古い図書館の閲覧席", "夜明けの砂漠", "ネオンの灯る路地", "静かな湖のほとり",
          "朝の市場", "廃線になった線路", "紅葉の山道", "夏祭りの神社の境内", "深夜のコンビニの前",
          "夕方の教室", "海沿いを走る電車の中", "古い喫茶店のカウンター", "雨の降るバス停", "花火大会の河川敷"]
PEOPLE = ["20代前半の黒髪ショートの女性", "10代後半の制服姿の少女", "30代の穏やかな表情の男性", "60代の優しい目をした女性",
          "20代後半のロングヘアの女性", "10代の自転車に乗った少年", "40代の疲れた表情のスーツ姿の男性", "20歳前後の青年",
          "70代の旅人の男性", "5歳くらいの女の子", "20代の浴衣姿の女性", "30代の赤いコートの女性"]
ACTIONS = ["ゆっくり歩きながら遠くを見つめる", "何かに気づいて振り返り微笑む", "空を見上げて手を伸ばす", "手紙を読んで涙ぐむ",
           "誰かに電話をかけながら小さく笑う", "息を切らして走り抜ける", "窓の外を見ながら静かに独り言を言う", "大切な人に向かって話しかける"]
MOODS = ["静かで穏やか", "切なくてノスタルジック", "幻想的", "わくわくする冒険の始まり",
         "あたたかく優しい", "少し不思議", "壮大で神秘的", "どこか寂しいけれど前向き"]
CAMERAS = ["カメラがゆっくり寄る", "低い位置から見上げる", "横に滑るように追う", "固定カメラで長めに見せる",
           "ゆっくり引いていく", "被写体を追う手持ちカメラ", "顔のクローズアップから引く"]

def _shot_times(seconds):
    n = 2 if seconds <= 5 else 3
    return [round(seconds * i / n, 3) for i in range(1, n)]

def _fmt_t(t):
    return "00:%06.3f" % t

def random_prompt_ja(seconds=5):
    seeds = dict(場所=random.choice(PLACES), 人物=random.choice(PEOPLE),
                 動作=random.choice(ACTIONS), 雰囲気=random.choice(MOODS), カメラ=random.choice(CAMERAS))
    user = "\n".join(k + "：" + v for k, v in seeds.items()) + f"\n動画の長さ：{seconds}秒"
    cuts = _shot_times(seconds)
    shots = "\n".join(["ショット1：…"] + [f"ショット{i+2}（{c:g}秒〜）：…" for i, c in enumerate(cuts)])
    system = ("あなたは映像作品の脚本家です。与えられた要素から、人物を主役にした短い動画の脚本を日本語で書きます。"
              "人物の年齢・髪型・服装・表情・仕草、場所の細部（光・天気・時間帯・質感）、カメラの動きを、"
              "ショットごとに具体的かつ細かく描写します。人物が日本語で話す短いセリフを必ず1〜2個入れます（「」でくくる）。"
              "次の形式だけを出力し、前置きや説明は書きません。\n"
              + shots + "\n環境音：…（足音・風・雨・物音など）\nBGM：…（楽器・テンポ・盛り上がり方）")
    return chat(system, user, max_new_tokens=700, temperature=0.9)

EXAMPLE_EN = (
    "integrated_multimodal_description:\n"
    "[Shot 1] Live-action, cinematic, shallow depth of field. A woman in her mid-twenties with a short black bob, wearing a beige trench coat, stands at a rain-wet bus stop at dusk, warm streetlight on her face. The camera pushes in slowly as she turns toward the road. The woman (S1), soft clear voice, gentle Tokyo accent, says: <d>[Japanese] もう、来ないのかな。</d>\n"
    "[Shot 2] At 00:03.000, close-up of her eyes catching the headlights of an approaching bus, a small smile forming. The camera pulls back slowly. She says: <d>[Japanese] ううん、大丈夫。</d>\n\n"
    "overall_soundscape: Steady rain on the shelter roof, distant traffic, tires hissing on wet asphalt, the low rumble of a bus slowing down.\n"
    "non_diegetic_music: Sparse felt-piano melody at a slow tempo with a soft sustained cello underneath, swelling gently toward the end.")

def translate_en(ja, seconds=5):
    cuts = _shot_times(seconds)
    cut_txt = "; ".join(f"[Shot {i+2}] starts at {_fmt_t(c)}" for i, c in enumerate(cuts))
    system = (
        "You write prompts for MiniMax H3, a video model that generates picture and sound together. "
        "Convert the Japanese scene description into this exact format and output nothing else:\n"
        "integrated_multimodal_description:\n[Shot 1] ...\n[Shot 2] At 00:SS.mmm, ...\n\n"
        "overall_soundscape: 1-4 English sentences of ambient and action sounds (no dialogue, no music).\n"
        "non_diegetic_music: 1-3 English sentences on audience-only background music (instruments, tempo, dynamics), or N/A.\n"
        "Rules: the first shot has no timestamp; later shots begin with 'At 00:SS.mmm,' and use exactly these cut times: " + cut_txt + ". "
        "Describe the person's age, hair, clothing, expression and gestures, the setting, lighting and the camera movement in detail, in English. "
        "Give each speaking person an ID like (S1) and introduce them with voice traits (age, pitch, timbre, accent) the first time. "
        "Put each spoken line in <d>[Japanese] line</d> keeping the Japanese words exactly as written. "
        "If the Japanese text lacks sound or music, invent fitting ones. Keep it in one continuous block.\n\n"
        "Example:\n" + EXAMPLE_EN)
    out = chat(system, f"動画の長さ：{seconds}秒\n" + ja, max_new_tokens=900, temperature=0.3)
    if "integrated_multimodal_description" not in out or "overall_soundscape" not in out:
        out = chat(system, f"動画の長さ：{seconds}秒\n" + ja + "\n\n必ず指定の形式（3つの項目）で出力してください。", max_new_tokens=900, temperature=0.2)
    if "non_diegetic_music" not in out:
        out += "\nnon_diegetic_music: N/A"
    return out

# ---- 状態 ----
T0 = time.time()
TOKEN = secrets.token_urlsafe(16)
URLS = {"comfy": None, "helper": None}

def build_status():
    elapsed_h = (time.time() - T0) / 3600
    remaining, source = None, "unavailable"
    if START_UNITS is not None and UNITS_PER_HOUR is not None:
        remaining, source = round(max(0.0, START_UNITS - elapsed_h * UNITS_PER_HOUR), 1), "estimate"
    return {
        "schema": 1,
        "updated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "comfy_url": URLS["comfy"],
        "helper_url": URLS["helper"],
        "helper_token": TOKEN,
        "gpu": {"name": GPU_NAME, "key": GPU_KEY, "vram_gb": round(VRAM_GB, 1), "sage": USE_SAGE},
        "usage": {"units_per_hour": UNITS_PER_HOUR, "units_remaining": remaining, "source": source},
        "models": {"diffusion": MODEL_FILES["diffusion"], "loras": []},
        "presets": [{"id": k, "label": v["label"], "max_sec": v["max_sec"].get(GPU_KEY),
                     "est_min": v["est_min"].get(GPU_KEY), "measured": v["measured"]} for k, v in PRESETS.items() if GPU_KEY in v["gpus"]],
        "frames": {str(s): snap_frames(s) for s in (3, 5, 10)},
    }

# ---- ヘルパーAPI ----
class JaIn(BaseModel):
    ja: str
    seconds: int = 5

class RandIn(BaseModel):
    seconds: int = 5

app = FastAPI()

def check(token):
    if token != TOKEN:
        raise HTTPException(401, "invalid token")

@app.get("/health")
def api_health():
    return {"ok": True}

@app.get("/status")
def api_status(x_token: str = Header(None)):
    check(x_token)
    return build_status()

@app.post("/random")
def api_random(body: RandIn = RandIn(), x_token: str = Header(None)):
    check(x_token)
    if comfy_busy():
        raise HTTPException(409, "生成中はプロンプトを作れません")
    ja = random_prompt_ja(body.seconds)
    return {"ja": ja, "en": translate_en(ja, body.seconds)}

@app.post("/translate")
def api_translate(body: JaIn, x_token: str = Header(None)):
    check(x_token)
    if comfy_busy():
        raise HTTPException(409, "生成中は変換できません")
    return {"en": translate_en(body.ja, body.seconds)}


# ---- 動画生成（ワークフローを組み立てて ComfyUI に送る）----
import base64 as _b64, copy as _copy, uuid as _uuid
TEMPLATE = json.loads(_b64.b64decode("eyIxMzEiOiB7ImlucHV0cyI6IHsic2NoZWR1bGVyIjogInNpbXBsZSIsICJzdGVwcyI6IDQsICJkZW5vaXNlIjogMSwgIm1vZGVsIjogWyIyMzAiLCAwXX0sICJjbGFzc190eXBlIjogIkJhc2ljU2NoZWR1bGVyIiwgIl9tZXRhIjogeyJ0aXRsZSI6ICIx5q6155uuIOOCueOCseOCuOODpeODvOODqSA0c3RlcCIsICJjbnJfaWQiOiAiY29tZnktY29yZSIsICJ2ZXIiOiAiMC4zOC4wIn19LCAiMTMyIjogeyJpbnB1dHMiOiB7Im5vaXNlIjogWyIxMzQiLCAwXSwgImd1aWRlciI6IFsiMTMzIiwgMF0sICJzYW1wbGVyIjogWyIyMDkiLCAwXSwgInNpZ21hcyI6IFsiMTMxIiwgMF0sICJsYXRlbnRfaW1hZ2UiOiBbIkdFTiIsIDFdfSwgImNsYXNzX3R5cGUiOiAiU2FtcGxlckN1c3RvbUFkdmFuY2VkIiwgIl9tZXRhIjogeyJ0aXRsZSI6ICIx5q6155uuIOOCteODs+ODl+ODqeODvCIsICJjbnJfaWQiOiAiY29tZnktY29yZSIsICJ2ZXIiOiAiMC4zOC4wIn19LCAiMTMzIjogeyJpbnB1dHMiOiB7Im1vZGVsIjogWyIyMzAiLCAwXSwgImNvbmRpdGlvbmluZyI6IFsiR0VOIiwgMF19LCAiY2xhc3NfdHlwZSI6ICJCYXNpY0d1aWRlciIsICJfbWV0YSI6IHsidGl0bGUiOiAi5Z+65pys44Ks44Kk44OA44O8IiwgImNucl9pZCI6ICJjb21meS1jb3JlIiwgInZlciI6ICIwLjM4LjAifX0sICIxMzQiOiB7ImlucHV0cyI6IHsibm9pc2Vfc2VlZCI6IDIwMjYxMDA2fSwgImNsYXNzX3R5cGUiOiAiUmFuZG9tTm9pc2UiLCAiX21ldGEiOiB7InRpdGxlIjogIuOCt+ODvOODiSjlm7rlrpopIiwgImNucl9pZCI6ICJjb21meS1jb3JlIiwgInZlciI6ICIwLjM4LjAifX0sICIxMzciOiB7ImlucHV0cyI6IHsiZXhwcmVzc2lvbiI6ICJtYXgoNSwgcm91bmQoYSAqIDI0KSkgKyAoNSAtIChtYXgoNSwgcm91bmQoYSAqIDI0KSkgJSAxNykpICUgMTciLCAidmFsdWVzLmEiOiBbIjE4MiIsIDBdfSwgImNsYXNzX3R5cGUiOiAiQ29tZnlNYXRoRXhwcmVzc2lvbiIsICJfbWV0YSI6IHsidGl0bGUiOiAi5pWw5byPIiwgImNucl9pZCI6ICJjb21meS1jb3JlIiwgInZlciI6ICIwLjM4LjAifX0sICIxNTMiOiB7ImlucHV0cyI6IHsidmFlX25hbWUiOiAibWluaW1heF9oM192aWRlb192YWVfaW50OF9jb252cm90LnNhZmV0ZW5zb3JzIn0sICJjbGFzc190eXBlIjogIlZBRUxvYWRlciIsICJfbWV0YSI6IHsidGl0bGUiOiAiVkFF44KS6Kqt44G/6L6844KAIiwgImNucl9pZCI6ICJjb21meS1jb3JlIiwgInZlciI6ICIwLjM4LjAifX0sICIxNTQiOiB7ImlucHV0cyI6IHsidmFlX25hbWUiOiAibWluaW1heF9oM19hdWRpb192YWVfZnAzMi5zYWZldGVuc29ycyJ9LCAiY2xhc3NfdHlwZSI6ICJWQUVMb2FkZXIiLCAiX21ldGEiOiB7InRpdGxlIjogIlZBReOCkuiqreOBv+i+vOOCgCIsICJjbnJfaWQiOiAiY29tZnktY29yZSIsICJ2ZXIiOiAiMC4zOC4wIn19LCAiMTY0IjogeyJpbnB1dHMiOiB7ImRldmljZSI6ICJkZWZhdWx0IiwgImNsaXAiOiBbIjIxMSIsIDBdfSwgImNsYXNzX3R5cGUiOiAiU2VsZWN0Q0xJUERldmljZSIsICJfbWV0YSI6IHsidGl0bGUiOiAiQ0xJUOODh+ODkOOCpOOCuemBuOaKniIsICJjbnJfaWQiOiAiY29tZnktY29yZSIsICJ2ZXIiOiAiMC4zOC4wIn19LCAiMTgxIjogeyJpbnB1dHMiOiB7ImFzcGVjdF9yYXRpbyI6ICI5OjE2IChQb3J0cmFpdCBXaWRlc2NyZWVuKSIsICJtZWdhcGl4ZWxzIjogMC41LCAibXVsdGlwbGUiOiAzMn0sICJjbGFzc190eXBlIjogIlJlc29sdXRpb25TZWxlY3RvciIsICJfbWV0YSI6IHsidGl0bGUiOiAi6Kej5YOP5bqmKDHmrrXnm64pIDk6MTYgLyAwLjVNUO+8iDU0NMOXOTYw77yJIiwgImNucl9pZCI6ICJjb21meS1jb3JlIiwgInZlciI6ICIwLjM4LjAifX0sICIxODIiOiB7ImlucHV0cyI6IHsidmFsdWUiOiA4fSwgImNsYXNzX3R5cGUiOiAiUHJpbWl0aXZlRmxvYXQiLCAiX21ldGEiOiB7InRpdGxlIjogIuenkuaVsCA477yI8J+OpOOCu+ODq+OBruenkuaVsOOBqOWQjOOBmOOBq+ODu+ODleODq0hE44GvIEw0IOOBpzjnp5Ljgb7jgafvvIkiLCAiY25yX2lkIjogImNvbWZ5LWNvcmUiLCAidmVyIjogIjAuMzguMCJ9fSwgIjE5MiI6IHsiaW5wdXRzIjogeyJlbmFibGVkIjogdHJ1ZSwgImJsZW5kX3dlaWdodCI6IDAuMywgImRlZ3JlZSI6IDEsICJyaWRnZV9sYW1iZGEiOiAwLjEsICJ3aW5kb3dfc2l6ZSI6IDIsICJmbGV4X3dpbmRvdyI6IDAuNzUsICJ3YXJtdXBfc3RlcHMiOiAxLCAidGFpbF9hY3R1YWxfc3RlcHMiOiAxLCAibWF4X2hpc3RvcnkiOiA4LCAiZGVidWciOiBmYWxzZSwgImhpc3Rvcnlfc3RvcmFnZSI6ICJzeXN0ZW1fcmFtIiwgImJvb3RzdHJhcF9maXJzdF9mb3JlY2FzdCI6IHRydWUsICJhbmNob3JfcmVzaWR1YWxfZmVlZGJhY2siOiBmYWxzZSwgInNlbGVjdGl2ZV9yb2xsYmFja19jb3JyZWN0aW9uIjogZmFsc2UsICJvZmZsaW5lX3Ntb290aGluZ19yZXBsYXkiOiB0cnVlLCAiYXVkaW9fYmxlbmRfd2VpZ2h0IjogMCwgIm9mZmxpbmVfYXJjaGl2ZV9zdG9yYWdlIjogInN5c3RlbV9yYW0iLCAibW9kZWxfYXdhcmVfbW9kZSI6ICJvZmYiLCAibW9kZWxfYXdhcmVfcmlza190aHJlc2hvbGQiOiAwLjY1LCAibW9kZWxfYXdhcmVfdHJ1c3Rfc2hyaW5rYWdlIjogZmFsc2UsICJtb2RlbF9hd2FyZV9yZXBsYXlfZ2VuZXJpY19jb3JyZWN0aW9uIjogZmFsc2UsICJnZW5lcmljX2NvcnJlY3Rpb25fbW9kZSI6ICJjb29yZGluYXRlX3JscyIsICJnZW5lcmljX2NvcnJlY3Rpb25fbGltaXRlciI6ICJoYXJkX2NsaXAiLCAiZ2VuZXJpY19jb3JyZWN0aW9uX2xpbWl0IjogMC40LCAiZ2VuZXJpY19jb3JyZWN0aW9uX2F0dGVudWF0aW9uIjogIm5vX2F0dGVudWF0aW9uIiwgInNhX3BlY2VfZm9yZWNhc3RfcG9saWN5IjogImJhbGFuY2VkIiwgIm1vZGVsIjogWyJMT1JBIiwgMF19LCAiY2xhc3NfdHlwZSI6ICJTcGVjdHJ1bUFwcGx5TWluaU1heEgzIiwgIl9tZXRhIjogeyJ0aXRsZSI6ICLpq5jpgJ/ljJY6IFNwZWN0cnVtIiwgImNucl9pZCI6ICJjb21meXVpLXNwZWN0cnVtLW1pbmltYXgtaDMiLCAiYXV4X2lkIjogInhtYXJyZS9Db21meVVJLVNwZWN0cnVtLU1pbmlNYXgtSDMiLCAidmVyIjogIjUxNjFmMDQ1N2JjOGM1MjUzNTIxMmQ2NzgzZWVlNzNmNDM5ZTE1MzcifX0sICIxOTMiOiB7ImlucHV0cyI6IHsibWV0aG9kIjogImxlYXJuZWQgbW9kZWwiLCAibGVhcm5lZF9tb2RlbCI6ICJoM19jbGVhbl9sYXRlbnRfdXBzY2FsZXJfZmlsbV9lcG9jaDIwMC5zYWZldGVuc29ycyIsICJhdWRpb19kZW5vaXNlIjogMCwgIm5vaXNlX3Jlc2FtcGxlIjogImluZGVwZW5kZW50IiwgIm1hdGNoX3N0YXRzIjogMCwgInNhbXBsZXMiOiBbIjEzMiIsIDFdLCAibW9kZWwiOiBbIjIzMCIsIDBdLCAibm9pc2UiOiBbIjEzNCIsIDBdLCAic2lnbWFzIjogWyIxOTkiLCAwXSwgInBvc2l0aXZlIjogWyJHRU4iLCAwXX0sICJjbGFzc190eXBlIjogIk1pbmlNYXhIM0xhdGVudFVwc2NhbGVDb21iaW5lZCIsICJfbWV0YSI6IHsidGl0bGUiOiAiMuauteebrjogTGF0ZW50IFVwc2NhbGVyIDJ4IiwgImF1eF9pZCI6ICJUcjFkYWUvQ29tZnlVSS1NaW5pTWF4SDNfTGF0ZW50VXBzY2FsZXIiLCAidmVyIjogIjg5NWUzYzQ3MTE2NDQyM2YwZWEwZThlYWY0NWViNzAxZWZlNjQxYWUifX0sICIxOTQiOiB7ImlucHV0cyI6IHsiYW1vdW50IjogMC4yLCAicmFkaXVzIjogMSwgInNhbXBsZXMiOiBbIjE5MyIsIDBdfSwgImNsYXNzX3R5cGUiOiAiTWluaU1heEgzTGF0ZW50U2hhcnBlbiIsICJfbWV0YSI6IHsidGl0bGUiOiAiMuauteebrjogTGF0ZW50IFNoYXJwZW4iLCAiYXV4X2lkIjogIlRyMWRhZS9Db21meVVJLU1pbmlNYXhIM19MYXRlbnRVcHNjYWxlciIsICJ2ZXIiOiAiODk1ZTNjNDcxMTY0NDIzZjBlYTBlOGVhZjQ1ZWI3MDFlZmU2NDFhZSJ9fSwgIjE5NSI6IHsiaW5wdXRzIjogeyJzaGlmdF92aWRlbyI6IDEyLCAic2hpZnRfYXVkaW8iOiAzLCAibW9kZWwiOiBbIjE5MiIsIDBdfSwgImNsYXNzX3R5cGUiOiAiTWluaU1heEgzU2lnbWFTaGlmdCIsICJfbWV0YSI6IHsidGl0bGUiOiAiU2lnbWFTaGlmdCIsICJjbnJfaWQiOiAiY29tZnktY29yZSIsICJ2ZXIiOiAiMC4zOC4wIn19LCAiMTk2IjogeyJpbnB1dHMiOiB7InVuZXRfbmFtZSI6ICJtaW5pbWF4X2gzX2Z1c2VkX3JlZmRlbHRhX3IxMDI0X3R1cmJvOF9teXN0aWMwN19pbnQ4X2NvbnZyb3Quc2FmZXRlbnNvcnMiLCAid2VpZ2h0X2R0eXBlIjogImRlZmF1bHQifSwgImNsYXNzX3R5cGUiOiAiVU5FVExvYWRlciIsICJfbWV0YSI6IHsidGl0bGUiOiAi44Oi44OH44OrOiBGdXNlZC1UdXJiby1JbnQ4IiwgImNucl9pZCI6ICJjb21meS1jb3JlIiwgInZlciI6ICIwLjM4LjAifX0sICIxOTgiOiB7ImlucHV0cyI6IHsibm9pc2UiOiBbIjIwMiIsIDBdLCAiZ3VpZGVyIjogWyIyMDAiLCAwXSwgInNhbXBsZXIiOiBbIjIwOSIsIDBdLCAic2lnbWFzIjogWyIxOTkiLCAwXSwgImxhdGVudF9pbWFnZSI6IFsiMTk0IiwgMF19LCAiY2xhc3NfdHlwZSI6ICJTYW1wbGVyQ3VzdG9tQWR2YW5jZWQiLCAiX21ldGEiOiB7InRpdGxlIjogIjLmrrXnm64g44K144Oz44OX44Op44O8IiwgImNucl9pZCI6ICJjb21meS1jb3JlIiwgInZlciI6ICIwLjM4LjAifX0sICIxOTkiOiB7ImlucHV0cyI6IHsic2NoZWR1bGVyIjogInNpbXBsZSIsICJzdGVwcyI6IDQsICJkZW5vaXNlIjogMC4zNSwgIm1vZGVsIjogWyIyMzAiLCAwXX0sICJjbGFzc190eXBlIjogIkJhc2ljU2NoZWR1bGVyIiwgIl9tZXRhIjogeyJ0aXRsZSI6ICIy5q6155uuIOOCueOCseOCuOODpeODvOODqSA0c3RlcCAvIGRlbm9pc2UwLjM177yI5LiL5pu444GN44GvIDLvvIkiLCAiY25yX2lkIjogImNvbWZ5LWNvcmUiLCAidmVyIjogIjAuMzguMCJ9fSwgIjIwMCI6IHsiaW5wdXRzIjogeyJtb2RlbCI6IFsiMjMwIiwgMF0sICJjb25kaXRpb25pbmciOiBbIjE5MyIsIDFdfSwgImNsYXNzX3R5cGUiOiAiQmFzaWNHdWlkZXIiLCAiX21ldGEiOiB7InRpdGxlIjogIuWfuuacrOOCrOOCpOODgOODvCIsICJjbnJfaWQiOiAiY29tZnktY29yZSIsICJ2ZXIiOiAiMC4zOC4wIn19LCAiMjAxIjogeyJpbnB1dHMiOiB7InRpbGluZyI6IHRydWUsICJ0aWxlX3NpemUiOiAyNTYsICJ0aWxlX292ZXJsYXAiOiA2NCwgIm91dHB1dF9kZXZpY2UiOiAiZ3B1IiwgInNhbXBsZXMiOiBbIjE5OCIsIDBdLCAidmFlIjogWyIxNTMiLCAwXX0sICJjbGFzc190eXBlIjogIk1pbmlNYXhIM1ZBRURlY29kZUZhc3QiLCAiX21ldGEiOiB7InRpdGxlIjogIjLmrrXnm646IEZhc3QgVkFFIERlY29kZSjjgr/jgqTjg6spIiwgImF1eF9pZCI6ICJUcjFkYWUvQ29tZnlVSS1NaW5pTWF4SDNfTGF0ZW50VXBzY2FsZXIiLCAidmVyIjogIjg5NWUzYzQ3MTE2NDQyM2YwZWEwZThlYWY0NWViNzAxZWZlNjQxYWUifX0sICIyMDIiOiB7ImlucHV0cyI6IHt9LCAiY2xhc3NfdHlwZSI6ICJEaXNhYmxlTm9pc2UiLCAiX21ldGEiOiB7InRpdGxlIjogIuODjuOCpOOCuuOCkueEoeWKueOBq+OBmeOCiyIsICJjbnJfaWQiOiAiY29tZnktY29yZSIsICJ2ZXIiOiAiMC4zOC4wIn19LCAiMjA4IjogeyJpbnB1dHMiOiB7InNhbXBsZXMiOiBbIjEzMiIsIDBdLCAidmFlIjogWyIxNTMiLCAwXX0sICJjbGFzc190eXBlIjogIlZBRURlY29kZSIsICJfbWV0YSI6IHsidGl0bGUiOiAiVkFF44OH44Kz44O844OJIiwgImNucl9pZCI6ICJjb21meS1jb3JlIiwgInZlciI6ICIwLjM4LjAifX0sICIyMDkiOiB7ImlucHV0cyI6IHsic2FtcGxlcl9uYW1lIjogInJlc19tdWx0aXN0ZXAifSwgImNsYXNzX3R5cGUiOiAiS1NhbXBsZXJTZWxlY3QiLCAiX21ldGEiOiB7InRpdGxlIjogIkvjgrXjg7Pjg5fjg6njg7zpgbjmip4iLCAiY25yX2lkIjogImNvbWZ5LWNvcmUiLCAidmVyIjogIjAuMzguMCJ9fSwgIjIxMCI6IHsiaW5wdXRzIjogeyJjbGlwX25hbWUiOiAicXdlbjN2bF80Yl9mcDhfc2NhbGVkLnNhZmV0ZW5zb3JzIiwgInR5cGUiOiAia3JlYTIiLCAiZGV2aWNlIjogImN1ZGE6MCIsICJtb2RlIjogImR5bmFtaWMifSwgImNsYXNzX3R5cGUiOiAiQ2xpcFByb2pEZXZpY2VMb2FkZXIiLCAiX21ldGEiOiB7InRpdGxlIjogIlRleHRFbmNvZGVyOiBDbGlwUHJvaig0QikiLCAiY25yX2lkIjogImNvbWZ5dWktY2xpcHByb2oiLCAiYXV4X2lkIjogIm5pY29sYWIyOC9Db21meVVJLUNsaXBQcm9qIiwgInZlciI6ICJjMDFiYThmYjhmNDFiNGYyMDk0ZGJkMGIxODVjZGMyMzhmYjYxMzRjIn19LCAiMjExIjogeyJpbnB1dHMiOiB7InByb2plY3Rpb24iOiAibW1oMy00Yi1DbGlwUHJvai12My4xLW1scC5zYWZldGVuc29ycyIsICJjbGlwIjogWyIyMTAiLCAwXX0sICJjbGFzc190eXBlIjogIkNsaXBQcm9qQXBwbHkiLCAiX21ldGEiOiB7InRpdGxlIjogIkNsaXBQcm9qIEFwcGx5IiwgImNucl9pZCI6ICJjb21meXVpLWNsaXBwcm9qIiwgImF1eF9pZCI6ICJuaWNvbGFiMjgvQ29tZnlVSS1DbGlwUHJvaiIsICJ2ZXIiOiAiYzAxYmE4ZmI4ZjQxYjRmMjA5NGRiZDBiMTg1Y2RjMjM4ZmI2MTM0YyJ9fSwgIjIzMCI6IHsiaW5wdXRzIjogeyJwcmVkaWN0b3IiOiAibWluaW1heF9oM19yMnZhX3ZlZGFfcHJldmlld19mcDguc2FmZXRlbnNvcnMiLCAiZ2VuZXJhdGVkX3NwYXJzaXR5IjogIjMyIiwgInJlZmVyZW5jZV9zcGFyc2l0eSI6ICIzMiIsICJmdWxsX2F0dGVudGlvbl9sYXllcnMiOiAiIiwgImZ1bGxfYXR0ZW50aW9uX3N0ZXBzIjogIiIsICJ2ZXJib3NlIjogZmFsc2UsICJtb2RlbCI6IFsiMTk1IiwgMF19LCAiY2xhc3NfdHlwZSI6ICJWZWRhU3BhcnNlQXR0ZW50aW9uIiwgIl9tZXRhIjogeyJ0aXRsZSI6ICLimqHpq5jpgJ/ljJY6IFZlZGHvvIhMNCDjgafntIQz5YCN6YCf44O755S76LOq44G744G85ZCM562J77yPQ3RybCtCIOOBp+OCquODle+8iSIsICJjbnJfaWQiOiAidmVkYS1zcGFyc2UtYXR0ZW50aW9uIiwgInZlciI6ICI4ZDdhYzE1M2FjMjE1ZDYzNmVhZjFiNTkxMGIwNTM2YzNkZTllNmVkIn19LCAiMjMzIjogeyJpbnB1dHMiOiB7ImZwcyI6IDI0LCAiYml0X2RlcHRoIjogImF1dG8iLCAiY29sb3Jfc3BhY2UiOiAic1JHQiIsICJjaHJvbWFfc3Vic2FtcGxpbmciOiAiYXV0byIsICJjb2RlYyI6ICJub25lIiwgImltYWdlcyI6IFsiMjAxIiwgMF19LCAiY2xhc3NfdHlwZSI6ICJDcmVhdGVWaWRlbyIsICJfbWV0YSI6IHsidGl0bGUiOiAi5a6M5oiQ5YuV55S7OiDmmKDlg4/vvIvimIXjgrvjg6rjg5Xpn7Plo7AiLCAiY25yX2lkIjogImNvbWZ5LWNvcmUiLCAidmVyIjogIjAuNDUuMCJ9fSwgIjIzNCI6IHsiaW5wdXRzIjogeyJmaWxlbmFtZV9wcmVmaXgiOiAiSVJPRE9SSS9rb2hhcnVfTDRfaXJvZG9yaSIsICJmb3JtYXQiOiAiYXV0byIsICJmb3JtYXQuY29kZWMiOiAiYXV0byIsICJjb2RlYyI6ICJhdXRvIiwgInZpZGVvIjogWyIyMzMiLCAwXX0sICJjbGFzc190eXBlIjogIlNhdmVWaWRlbyIsICJfbWV0YSI6IHsidGl0bGUiOiAi4piF5a6M5oiQ5YuV55S777yI5L+d5a2Y77ya5bem44Gu44CM44Ki44K744OD44OI44CN4oaSIOWLleeUu+OBruOAjOKApuOAjeKGkiDjg4Djgqbjg7Pjg63jg7zjg4nvvIkiLCAiY25yX2lkIjogImNvbWZ5LWNvcmUiLCAidmVyIjogIjAuNDUuMCJ9fX0=").decode())   # L4実測の「2段生成＋Veda」ワークフロー（API形式）

def _opts(spec):
    # 選択肢の取り出し（旧形式 [[...]] と新形式 ["COMBO", {"options": [...]}] の両対応）
    if isinstance(spec, list) and spec:
        if isinstance(spec[0], list):
            return spec[0]
        if len(spec) > 1 and isinstance(spec[1], dict) and "options" in spec[1]:
            return spec[1]["options"]
    return []

def _info(cls):
    r = requests.get(COMFY + "/object_info/" + cls, timeout=30).json()[cls]["input"]
    d = dict(r.get("required", {})); d.update(r.get("optional", {}))
    return d

def _fill_required(cls, ins):
    # 必須入力で指定していないものを、ノードの標準値で埋める
    req = requests.get(COMFY + "/object_info/" + cls, timeout=30).json()[cls]["input"].get("required", {})
    for k, v in req.items():
        if k in ins:
            continue
        t = v[0]
        o = v[1] if len(v) > 1 and isinstance(v[1], dict) else {}
        if "default" in o:
            ins[k] = o["default"]
        elif isinstance(t, list) and t:
            ins[k] = t[0]
        elif t == "STRING":
            ins[k] = ""
        elif t == "BOOLEAN":
            ins[k] = False
        elif t in ("INT", "FLOAT"):
            ins[k] = o.get("min", 0)
    return ins

def _upload(b64, name):
    files = {"image": (name, _b64.b64decode(b64), "image/png")}
    r = requests.post(COMFY + "/upload/image", files=files, data={"overwrite": "true"}, timeout=120)
    r.raise_for_status()
    return r.json()["name"]

class GenIn(BaseModel):
    mode: str = "t2v"            # t2v / i2v / ref2v
    prompt_en: str
    seconds: float = 5
    aspect: str = "9:16"         # 9:16 / 16:9 / 1:1
    preset: str = "standard"     # PRESETS のキー
    seed: int = 0
    images_b64: list = []        # i2v は1枚、ref2v は最大3枚
    lora: str = ""               # 空ならなし
    lora_strength: float = 1.0
    checkpoint: str = ""         # 空なら標準のH3。diffusion_models内のファイル名
    client_id: str = ""          # アプリのWebSocket接続ID（進捗をそのIDに届ける）
    refmods: list = []           # [{"name": "...", "strength": 1.0}]（RefMod。最大8）

def build_graph(g):
    if g.preset not in PRESETS or GPU_KEY not in PRESETS[g.preset]["gpus"]:
        raise HTTPException(400, "このGPUでは使えない画質です")
    p = PRESETS[g.preset]
    max_sec = p["max_sec"].get(GPU_KEY, 5)
    if g.seconds > max_sec:
        raise HTTPException(400, f"このGPU・画質では最大 {max_sec} 秒です")
    w = _copy.deepcopy(TEMPLATE)
    # 音声: 2段目の映像+音声ラテントから音声を取り出して動画にまとめる
    w["AS"] = {"class_type": "LTXVSeparateAVLatent", "inputs": {"av_latent": ["198", 0]}}
    w["AD"] = {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["AS", 1], "vae": ["154", 0]}}
    w["233"]["inputs"]["audio"] = ["AD", 0]
    w["134"]["inputs"]["noise_seed"] = int(g.seed)
    w["182"]["inputs"]["value"] = float(g.seconds)
    w["181"]["inputs"]["megapixels"] = p["stage1_mp"]
    opts = _opts(_info("ResolutionSelector")["aspect_ratio"])
    w["181"]["inputs"]["aspect_ratio"] = next((o for o in opts if o.startswith(g.aspect)), opts[0])

    # 生成ノード（モードで切り替え）
    cls = "MiniMaxH3ReferenceToVideo" if g.mode == "ref2v" else "MiniMaxH3ImageToVideo"
    spec = _info(cls)
    node = {"prompt": g.prompt_en, "width": ["181", 0], "height": ["181", 1], "length": ["137", 1],
            "clip": ["164", 0], "vae": ["153", 0], "audio_vae": ["154", 0]}
    if "ref_image_size" in spec:
        node["ref_image_size"] = "match"
    imgs = [_upload(b, f"app_{_uuid.uuid4().hex[:8]}_{i}.png") for i, b in enumerate(g.images_b64)]
    need = {"t2v": 0, "i2v": 1, "ref2v": 1}[g.mode]
    if len(imgs) < need:
        raise HTTPException(400, "画像が足りません")
    if g.mode == "i2v":
        imgs = imgs[:1]
    for i, name in enumerate(imgs[:9]):
        lid = f"IMG{i}"
        w[lid] = {"class_type": "LoadImage", "inputs": {"image": name}}
        if g.mode == "i2v":
            node["first_frame"] = [lid, 0]          # 画像→動画: 1枚目を最初のフレームに
        else:
            node[f"ref_images.ref_image_{i}"] = [lid, 0]   # 参照画像→動画: 参照画像を順に渡す
    if g.mode == "ref2v" and not g.prompt_en.lstrip().startswith(("subject_definitions", "<Picture")):
        tags = " ".join(f"<Picture {i+1}> is reference image {i+1}." for i in range(len(imgs)))
        node["prompt"] = tags + " " + g.prompt_en
    node = {k: v for k, v in node.items() if k in spec or k.startswith("ref_images.")}
    w["GEN"] = {"class_type": cls, "inputs": node}

    # RefMod（保存済みの参照をキャラ固定に使う）
    if g.refmods:
        have = set(list_refmods())
        rows = []
        for m in g.refmods[:8]:
            n = str(m.get("name", ""))
            if n not in have:
                raise HTTPException(400, "そのRefModが見つかりません: " + n)
            rows.append((n, float(m.get("strength", 1.0))))
        li = {"show_info": False}
        for i in range(1, 9):
            n, st_ = rows[i - 1] if i <= len(rows) else ("(none)", 1.0)
            li[f"mod_{i}"] = n; li[f"strength_{i}"] = st_; li[f"copies_{i}"] = 1
        _fill_required("MiniMaxH3RefModsLoader", li)
        w["MODL"] = {"class_type": "MiniMaxH3RefModsLoader", "inputs": li}
        w["MODA"] = {"class_type": "MiniMaxH3RefModApply", "inputs": _fill_required("MiniMaxH3RefModApply", {
            "conditioning": ["GEN", 0], "mods": ["MODL", 0], "override": False, "retention": 1.0,
            "curve_direction": "constant", "scramble_seed": -1, "curve_shape": "linear", "curve_value": 1.0})}

    # LoRA
    if g.lora:
        w["LORA"] = {"class_type": "LoraLoaderModelOnly",
                     "inputs": {"model": ["196", 0], "lora_name": g.lora, "strength_model": g.lora_strength}}
    else:
        w["LORA"] = None

    # 参照を解決（GEN / LORA の仮名を本物の接続先に）
    def fix(v, nid=""):
        if isinstance(v, list) and len(v) == 2 and v[0] == "LORA" and w["LORA"] is None:
            return ["196", 0]
        if isinstance(v, list) and len(v) == 2 and v[0] == "GEN" and v[1] == 0 and "MODA" in w and nid != "MODA":
            return ["MODA", 0]     # RefModを通した条件付けに差し替える
        return v
    for nid, n in list(w.items()):
        if n is None:
            continue
        n["inputs"] = {k: fix(v, nid) for k, v in n["inputs"].items()}
    w.pop("LORA") if w["LORA"] is None else None
    if g.checkpoint:
        if g.checkpoint not in list_checkpoints():
            raise HTTPException(400, "そのチェックポイントは見つかりません")
        w["196"]["inputs"]["unet_name"] = g.checkpoint
    w["234"]["inputs"]["filename_prefix"] = "app/video"
    return w

@app.post("/generate")
def api_generate(g: GenIn, x_token: str = Header(None)):
    check(x_token)
    graph = build_graph(g)
    r = requests.post(COMFY + "/prompt", json={"prompt": graph, "client_id": (g.client_id or _uuid.uuid4().hex)}, timeout=60)
    if not r.ok:
        raise HTTPException(500, "ComfyUIが受け付けませんでした: " + r.text[:600])
    return r.json()   # prompt_id を返す。進捗は ComfyUI の /ws、完成は /history/{prompt_id} と /view で取得

@app.get("/loras")
def api_loras(x_token: str = Header(None)):
    check(x_token)
    return {"loras": list(_opts(_info("LoraLoaderModelOnly")["lora_name"]))}


# ---- RefMod（参照画像から作る小さな参照ファイル。LoRAのように選んで使う）----
REFMOD_DIR = f"{C}/models/refmods"
DRIVE_REFMOD_DIR = os.path.join(DRIVE_DIR, "refmods")
os.makedirs(REFMOD_DIR, exist_ok=True)
os.makedirs(DRIVE_REFMOD_DIR, exist_ok=True)
for _f in os.listdir(DRIVE_REFMOD_DIR):      # 前回までに作ったものをドライブから戻す
    if _f.endswith(".safetensors") and not os.path.exists(os.path.join(REFMOD_DIR, _f)):
        shutil.copyfile(os.path.join(DRIVE_REFMOD_DIR, _f), os.path.join(REFMOD_DIR, _f))
REFMOD_JOBS = {}

def list_refmods():
    out = []
    for root, _, files in os.walk(REFMOD_DIR):
        for f in files:
            if f.endswith(".safetensors"):
                rel = os.path.relpath(os.path.join(root, f), REFMOD_DIR)
                out.append(rel[:-len(".safetensors")].replace(os.sep, "/"))
    return sorted(out)

class RefModIn(BaseModel):
    name: str
    images_b64: list
    detail: str = "standard"      # standard=512 / high=768

def _refmod_job(job, b: RefModIn):
    try:
        before = set(os.listdir(REFMOD_DIR))
        imgs = [_upload(x, f"refmod_{_uuid.uuid4().hex[:8]}_{i}.png") for i, x in enumerate(b.images_b64[:8])]
        spec = _info("MiniMaxH3RefModExtract")
        modes = _opts(spec.get("mode", []))
        ctypes = _opts(spec.get("concept_type", []))
        ins = {"name": b.name, "vae": ["V", 0],
               "mode": "Full Reference" if "Full Reference" in modes else "encode",
               "concept_type": "identity" if "identity" in ctypes else (ctypes[0] if ctypes else "generic"),
               "ref_resolution": 768 if b.detail == "high" else 512, "max_tokens": 0}
        w = {"V": {"class_type": "VAELoader", "inputs": {"vae_name": MODEL_FILES["video_vae"]}}}
        for i, nm in enumerate(imgs):
            w[f"I{i}"] = {"class_type": "LoadImage", "inputs": {"image": nm}}
            ins[f"refs_image.ref_image_{i}"] = [f"I{i}", 0]
        ins["save"] = True
        ins = {k: v for k, v in ins.items() if k in spec or k.startswith("refs_image.")}
        _fill_required("MiniMaxH3RefModExtract", ins)
        w["X"] = {"class_type": "MiniMaxH3RefModExtract", "inputs": ins}
        r = requests.post(COMFY + "/prompt", json={"prompt": w, "client_id": "refmod"}, timeout=60)
        if not r.ok:
            raise RuntimeError(r.text[:500])
        pid = r.json()["prompt_id"]
        for _ in range(900):
            h = requests.get(COMFY + "/history/" + pid, timeout=30).json()
            if pid in h:
                st_ = h[pid].get("status", {})
                if st_.get("status_str") == "error":
                    raise RuntimeError("ComfyUIでエラーになりました")
                break
            time.sleep(2)
        else:
            raise RuntimeError("時間がかかりすぎたので中止しました")
        new = [f for f in set(os.listdir(REFMOD_DIR)) - before if f.endswith(".safetensors")]
        if not new:
            raise RuntimeError("RefModのファイルができませんでした")
        for f in new:
            shutil.copyfile(os.path.join(REFMOD_DIR, f), os.path.join(DRIVE_REFMOD_DIR, f))
        job["name"] = new[0][:-len(".safetensors")]
        job["state"] = "done"
    except Exception as e:
        job["state"] = "error"; job["error"] = str(e)

@app.get("/refmods")
def api_refmods(x_token: str = Header(None)):
    check(x_token)
    return {"refmods": list_refmods()}

@app.post("/refmods")
def api_refmod_create(b: RefModIn, x_token: str = Header(None)):
    check(x_token)
    nm = re.sub(r"[^0-9A-Za-z_\-]", "_", b.name.strip())[:40]
    if not nm:
        raise HTTPException(400, "名前は英数字で入れてください")
    if not b.images_b64:
        raise HTTPException(400, "画像を1枚以上えらんでください")
    b.name = nm
    jid = _uuid.uuid4().hex[:8]
    job = {"state": "running", "name": nm, "error": ""}
    REFMOD_JOBS[jid] = job
    threading.Thread(target=_refmod_job, args=(job, b), daemon=True).start()
    return {"id": jid}

@app.get("/refmods/job/{jid}")
def api_refmod_job(jid: str, x_token: str = Header(None)):
    check(x_token)
    if jid not in REFMOD_JOBS:
        raise HTTPException(404, "not found")
    return REFMOD_JOBS[jid]

@app.delete("/refmods")
def api_refmod_delete(name: str, x_token: str = Header(None)):
    check(x_token)
    rel = name.replace("..", "") + ".safetensors"
    for base in (REFMOD_DIR, DRIVE_REFMOD_DIR):
        pth = os.path.join(base, rel)
        if os.path.exists(pth):
            os.remove(pth)
    return {"ok": True}

# ---- モデル管理（チェックポイント一覧・LoRA追加ダウンロード）----
import urllib.parse as _up
MODEL_DIRS = {"lora": f"{C}/models/loras", "checkpoint": f"{C}/models/diffusion_models"}
ALLOWED_HOSTS = ("huggingface.co", "hf.co", "civitai.com", "civitai.red")
DOWNLOADS = {}

def list_checkpoints():
    d = MODEL_DIRS["checkpoint"]
    return sorted(f for f in os.listdir(d) if f.endswith((".safetensors", ".gguf"))) if os.path.isdir(d) else []

class DlIn(BaseModel):
    url: str
    kind: str = "lora"           # lora / checkpoint
    filename: str = ""
    token: str = ""              # 必要な場合だけ（CivitAI等のAPIキー、HuggingFaceのトークン）

def _download(job, url, dest, token):
    try:
        headers = {}
        host = _up.urlparse(url).hostname or ""
        if token:
            if host.endswith("civitai.com") or host.endswith("civitai.red"):
                url += ("&" if "?" in url else "?") + "token=" + token
            else:
                headers["Authorization"] = "Bearer " + token
        tmp = dest + ".part"
        with requests.get(url, headers=headers, stream=True, timeout=60, allow_redirects=True) as r:
            if not r.ok:
                raise RuntimeError(f"ダウンロード元が応じませんでした (HTTP {r.status_code})。トークンが必要な場合があります")
            job["total"] = int(r.headers.get("content-length") or 0)
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
                    job["bytes"] += len(chunk)
        if job["bytes"] < 1 << 20:
            raise RuntimeError("ファイルが小さすぎます。URLがモデルのファイルを指しているか確認してください")
        os.replace(tmp, dest)
        job["state"] = "done"
    except Exception as e:
        job["state"] = "error"; job["error"] = str(e)
        try: os.remove(dest + ".part")
        except Exception: pass

@app.get("/models")
def api_models(x_token: str = Header(None)):
    check(x_token)
    return {"checkpoints": list_checkpoints(), "default": MODEL_FILES["diffusion"],
            "loras": list(_opts(_info("LoraLoaderModelOnly")["lora_name"]))}

@app.post("/download")
def api_download(b: DlIn, x_token: str = Header(None)):
    check(x_token)
    u = _up.urlparse(b.url.strip())
    if u.scheme != "https" or not any((u.hostname or "").endswith(h) for h in ALLOWED_HOSTS):
        raise HTTPException(400, "HuggingFace か CivitAI のURLだけ使えます")
    if b.kind not in MODEL_DIRS:
        raise HTTPException(400, "種類が違います")
    name = os.path.basename(b.filename.strip() or _up.unquote(u.path))
    name = re.sub(r"[^A-Za-z0-9._-]", "_", name)
    if not name.endswith(".safetensors"):
        name = (name or "model") + (".safetensors" if not name.endswith(".gguf") else "")
    os.makedirs(MODEL_DIRS[b.kind], exist_ok=True)
    dest = os.path.join(MODEL_DIRS[b.kind], name)
    if os.path.exists(dest):
        raise HTTPException(409, "同じ名前のファイルがもうあります: " + name)
    job = {"id": _uuid.uuid4().hex[:8], "name": name, "kind": b.kind, "state": "running", "bytes": 0, "total": 0, "error": ""}
    DOWNLOADS[job["id"]] = job
    threading.Thread(target=_download, args=(job, b.url.strip(), dest, b.token.strip()), daemon=True).start()
    return job

@app.get("/download/{jid}")
def api_download_status(jid: str, x_token: str = Header(None)):
    check(x_token)
    if jid not in DOWNLOADS:
        raise HTTPException(404, "not found")
    return DOWNLOADS[jid]

@app.delete("/models")
def api_models_delete(kind: str, name: str, x_token: str = Header(None)):
    check(x_token)
    if kind != "lora":
        raise HTTPException(400, "LoRAだけ削除できます")
    p = os.path.join(MODEL_DIRS["lora"], os.path.basename(name))
    if os.path.exists(p):
        os.remove(p)
    return {"ok": True}

# ---- Googleドライブのフォルダから取り込む ----
# ドライブの ColabVideoApp/loras と ColabVideoApp/checkpoints に置いたファイルを、Colabのローカルへコピーして使う
import shutil
def _copy_job(job, src, dest):
    try:
        job["total"] = os.path.getsize(src)
        tmp = dest + ".part"
        with open(src, "rb") as fi, open(tmp, "wb") as fo:
            while True:
                b = fi.read(8 << 20)
                if not b:
                    break
                fo.write(b)
                job["bytes"] += len(b)
        os.replace(tmp, dest)
        job["state"] = "done"
    except Exception as e:
        job["state"] = "error"; job["error"] = str(e)
        try: os.remove(dest + ".part")
        except Exception: pass

class ImportIn(BaseModel):
    kind: str
    name: str

@app.get("/drive/list")
def api_drive_list(x_token: str = Header(None)):
    check(x_token)
    out = {}
    for kind, d in DRIVE_MODEL_DIRS.items():
        items = []
        for f in sorted(os.listdir(d)) if os.path.isdir(d) else []:
            if f.endswith((".safetensors", ".gguf")):
                items.append({"name": f, "size": os.path.getsize(os.path.join(d, f)),
                              "imported": os.path.exists(os.path.join(MODEL_DIRS[kind], f))})
        out[kind] = items
    out["folder"] = "マイドライブ/ColabVideoApp/（loras / checkpoints）"
    return out

@app.post("/drive/import")
def api_drive_import(b: ImportIn, x_token: str = Header(None)):
    check(x_token)
    if b.kind not in DRIVE_MODEL_DIRS:
        raise HTTPException(400, "種類が違います")
    name = os.path.basename(b.name)
    src = os.path.join(DRIVE_MODEL_DIRS[b.kind], name)
    if not os.path.exists(src):
        raise HTTPException(404, "ドライブにそのファイルがありません")
    os.makedirs(MODEL_DIRS[b.kind], exist_ok=True)
    dest = os.path.join(MODEL_DIRS[b.kind], name)
    if os.path.exists(dest):
        raise HTTPException(409, "もう取り込み済みです")
    job = {"id": _uuid.uuid4().hex[:8], "name": name, "kind": b.kind, "state": "running", "bytes": 0, "total": 0, "error": ""}
    DOWNLOADS[job["id"]] = job
    threading.Thread(target=_copy_job, args=(job, src, dest), daemon=True).start()
    return job

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
print("ヘルパーAPIが起動しました（port 8000）")

In [ ]:
#@title 6. トンネルを開く（アプリから届くURLを作る）
CF = "/content/cloudflared"
if not os.path.exists(CF):
    sh(f"wget -q -O {CF} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {CF}")

def open_tunnel(port):
    log = f"/content/cloudflared_{port}.log"
    proc = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}"],
                            stdout=open(log, "w"), stderr=subprocess.STDOUT)
    for _ in range(90):
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(log).read())
        if m:
            return proc, m.group(0)
        time.sleep(1)
    raise RuntimeError("トンネルのURLを取得できませんでした: " + log)

tunnel_comfy, URLS["comfy"] = open_tunnel(8188)
tunnel_helper, URLS["helper"] = open_tunnel(8000)
print("ComfyUI :", URLS["comfy"])
print("ヘルパー:", URLS["helper"])

In [ ]:
#@title 7. アプリに状態を知らせ続ける（このセルは止めないでください）
def write_status(state="running"):
    data = build_status()
    data["state"] = state
    tmp = STATUS_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, STATUS_PATH)

print("ドライブに書き出します:", STATUS_PATH)
print("アプリの接続タブで「接続テスト」を押してください。止めるときはこのセルを停止します。")
try:
    while True:
        state = "running" if comfy_proc.poll() is None else "error"
        write_status(state)
        time.sleep(30)
except KeyboardInterrupt:
    pass
finally:
    write_status("stopped")
    print("停止しました")